# SPS-CA Colab

Run the current `main` repository, keep Ollama/Web UI, and test the existing 1000-scenario suite without generating new scenarios.

In [6]:
# CELL 1 — ENVIRONMENT + OLLAMA
import os, subprocess, sys, time
#MODEL = 'qwen2.5-coder:7b'
MODEL = 'qwen2.5-coder:14b'
#MODEL = 'qwen3-coder:30b'
def run(cmd, check=True):
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout)
    if check and r.returncode: raise RuntimeError(r.stdout)
run('apt-get update -qq')
run('apt-get install -y -qq git curl zstd')
run(f'{sys.executable} -m pip install -q --upgrade pip')
run(f'{sys.executable} -m pip install -q pyngrok pytest pytest-json-report')
if subprocess.run('command -v ollama', shell=True, capture_output=True).returncode != 0: run('curl -fsSL https://ollama.com/install.sh | sh')
run('ollama --version')
subprocess.run("pkill -f 'ollama serve' 2>/dev/null || true", shell=True)
subprocess.Popen(['ollama','serve'], stdout=open('/tmp/ollama.log','w'), stderr=subprocess.STDOUT)
for _ in range(30):
    time.sleep(1)
    if subprocess.run('curl -s http://127.0.0.1:11434/api/tags', shell=True, capture_output=True).returncode == 0: break
else: raise RuntimeError('Ollama service did not become ready')
run(f'ollama pull {MODEL}')
run('ollama list')
print('CELL 1 PASSED')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)




ollama version is 0.33.3

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling ac9bc7a69dab:   0% ▕                  ▏  34 MB/9.0 GB                  pulling manifest 
pulling ac9bc7a69dab:   1% ▕                  ▏  61 MB/9.0 GB                  pulling manifest 
pulling ac9bc7a69dab:   1% ▕                  ▏ 125 MB/9.0 GB                  pulling manifes

In [2]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

BASE = Path("/content")
REPO = BASE / "SPS_CA"
REMOTE = "https://github.com/muhammadnaumantahir/SPS_CA.git"

# Important: never delete the directory we are currently inside.
os.chdir(BASE)

if REPO.exists():
    shutil.rmtree(REPO)

result = subprocess.run(
    ["git", "clone", "--branch", "main", "--single-branch", REMOTE, str(REPO)],
    text=True,
    capture_output=True,
)

print("Git return code:", result.returncode)
if result.stdout:
    print("STDOUT:\n", result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"Git clone failed with code {result.returncode}")

os.chdir(REPO)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)

print("Pulled:", subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    text=True
).strip())

Git return code: 0
STDERR:
 Cloning into '/content/SPS_CA'...

Pulled: 0e4cf71


In [3]:
# CELL 3 — CORE SMOKE CHECK
import os, sys
os.chdir('/content/SPS_CA')
sys.path.insert(0,'/content/SPS_CA') if '/content/SPS_CA' not in sys.path else None
from layers.architecture import architecture_manifest
from core.canonical_sps_pipeline import CanonicalSPSPipeline
from layers.layer_08_evolution_core import GrowthDecisionEngine
m = architecture_manifest()
assert len(m['layers']) == 10 and m['brain']['name'] == 'SPS-CA Brain'
print('✓ Ten layers, Brain, pipeline, and Layer 8 available')

✓ Ten layers, Brain, pipeline, and Layer 8 available


In [4]:
# CELL 4 — SPS-CA WEB UI + NGROK
import os, subprocess, sys, time
from getpass import getpass
from pyngrok import ngrok
os.chdir('/content/SPS_CA'); PORT = 5000
subprocess.run("pkill -f 'ui.web_app' 2>/dev/null || true", shell=True)
log = open('/tmp/sps_ca_web_ui.log','w')
server = subprocess.Popen([sys.executable,'-m','ui.web_app'], cwd='/content/SPS_CA', stdout=log, stderr=subprocess.STDOUT)
time.sleep(2)
token = "3IlUVhidvYsN4j8cllrERLlyZMd_6xaqLH3PGtZwYR7St8sHv"
if not token:
    try: token = getpass('Enter NGROK_AUTHTOKEN (blank = local UI): ').strip()
    except Exception: token = ''
if token:
    ngrok.set_auth_token(token); tunnel = ngrok.connect(PORT,'http'); print('SPS-CA public Web UI:', tunnel.public_url)
else: print('SPS-CA local Web UI: http://127.0.0.1:5000')
print('✓ Web UI started'); print('✓ Ollama: http://127.0.0.1:11434')

SPS-CA public Web UI: https://unwired-bucket-surely.ngrok-free.dev
✓ Web UI started
✓ Ollama: http://127.0.0.1:11434


In [5]:
import requests, os, subprocess

print("=== SPS-CA ===")
print(requests.get("http://127.0.0.1:5000/api/capabilities", timeout=10).status_code)

print("\n=== OLLAMA ===")
r = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)
print(r.status_code)
print([m.get("name") for m in r.json().get("models", [])])

print("\n=== WEB LOG ===")
print(subprocess.run(
    ["bash", "-lc", "tail -n 80 /tmp/sps_ca_web_ui.log"],
    capture_output=True, text=True
).stdout)

=== SPS-CA ===
200

=== OLLAMA ===
200
['qwen2.5-coder:7b']

=== WEB LOG ===



In [6]:
import requests
import json
import time

BASE = "http://127.0.0.1:5000"

print("=== 1. CREATE SESSION ===")
r = requests.post(
    f"{BASE}/api/sessions",
    json={"title": "stream diagnostic"},
    timeout=10,
)
print("status:", r.status_code)
print("response:", r.text)

session_id = r.json()["id"]

print("\n=== 2. TEST CHAT STREAM ===")
payload = {
    "session_id": session_id,
    "request": "Generate a Python function called add(a, b) that returns a + b.",
    "code": "",
    "filename": "main.py",
    "model": "qwen3-coder:30b",
    "conversation": [],
}

started = time.time()

try:
    with requests.post(
        f"{BASE}/api/chat/stream",
        json=payload,
        stream=True,
        timeout=(10, None),   # connect timeout 10s, model generation unlimited
    ) as r:
        print("HTTP status:", r.status_code)
        print("Content-Type:", r.headers.get("content-type"))
        print("Headers:", dict(r.headers))

        for line in r.iter_lines(decode_unicode=True):
            if line:
                print(f"[{time.time()-started:6.1f}s] {line}")

except Exception as e:
    print("STREAM ERROR:", repr(e))

print("\n=== 3. WEB LOG ===")
import subprocess
print(
    subprocess.run(
        ["bash", "-lc", "cat /tmp/sps_ca_web_ui.log"],
        capture_output=True,
        text=True,
    ).stdout
)

=== 1. CREATE SESSION ===
status: 201
response: {"id": "7332fa739210", "title": "stream diagnostic", "created_at": "2026-09-04T12:06:23.940518+00:00", "updated_at": "2026-09-04T12:06:23.940535+00:00", "conversation": [], "code": "", "filename": "main.py", "detected_language": "unknown", "language_confidence": 0.0, "model": ""}

=== 2. TEST CHAT STREAM ===
HTTP status: 200
Content-Type: text/event-stream; charset=utf-8
Headers: {'Server': 'SPS-CA/3.6 Python/3.13.15', 'Date': 'Fri, 04 Sep 2026 12:06:23 GMT', 'Content-Type': 'text/event-stream; charset=utf-8', 'Cache-Control': 'no-cache, no-store, must-revalidate', 'Connection': 'close', 'X-Accel-Buffering': 'no'}
[   4.8s] data: {"type": "stage", "stage": "request_received", "progress": 8}
[   4.8s] data: {"type": "stage", "stage": "running", "progress": 22, "elapsed_seconds": 0.8}
[   4.8s] data: {"type": "stage", "stage": "running", "progress": 27, "elapsed_seconds": 1.6}
[   4.8s] data: {"type": "stage", "stage": "running", "progress"

KeyboardInterrupt: 

In [ ]:
# CELL 5 — TEST EXISTING 1000 SCENARIOS (NO GENERATION)
import json, subprocess, sys
from pathlib import Path
REPO = Path('/content/SPS_CA'); SCENARIO_FILE = REPO/'evaluation/scenarios/growth.json'; REPORT_FILE = Path('/content/sps_ca_1000_test_report.json')
assert SCENARIO_FILE.exists(), SCENARIO_FILE
from scripts.evaluate_growth_scenarios import load, validate_structure, validate_routing, validate_evolution_contracts
scenarios = load(); validate_structure(scenarios); validate_routing(scenarios); validate_evolution_contracts(scenarios)
r = subprocess.run([sys.executable,'-m','pytest','layers/layer_08_evolution_core/tests/test_growth_decision_scoring.py','layers/layer_08_evolution_core/tests/test_capability_improvement.py','-q'], cwd=REPO, text=True, capture_output=True)
print(r.stdout)
if r.returncode: print(r.stderr); raise RuntimeError('Targeted evolution tests failed')
breakdown = {}
for s in scenarios: breakdown[s['scenario_type']] = breakdown.get(s['scenario_type'],0)+1
report = {'status':'PASS','total':len(scenarios),'breakdown':breakdown,'generation_called':False,'targeted_evolution_tests':'PASS','scenario_file':str(SCENARIO_FILE)}
REPORT_FILE.write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
print(f"✓ Existing suite tested: {len(scenarios)}/1000")
print(f"✓ Routing: {breakdown.get('capability_routing',0)}/490")
print(f"✓ Autonomous evolution: {breakdown.get('autonomous_evolution',0)}/500")
print(f"✓ Evolution proof: {breakdown.get('evolution_proof',0)}/10")
print('✓ No scenario generation called')

In [ ]:
# CELL 6 — REPORT RESULTS
import json
from pathlib import Path
r = json.loads(Path('/content/sps_ca_1000_test_report.json').read_text(encoding='utf-8')); b = r['breakdown']
print('='*70); print('SPS-CA 1000-SCENARIO REPORT'); print('='*70)
print('Overall:', r['status']); print('Total:', f"{r['total']}/1000")
print('Capability routing:', f"{b.get('capability_routing',0)}/490")
print('Autonomous evolution:', f"{b.get('autonomous_evolution',0)}/500")
print('Evolution proof:', f"{b.get('evolution_proof',0)}/10")
print('Scenario generation called:', r['generation_called'])
print('Targeted scoring/improvement tests:', r['targeted_evolution_tests'])
print('='*70)

In [ ]:
from pathlib import Path
import json

from layers.layer_08_evolution.evolution_evidence import EvolutionEvidenceStore

demo_root = Path("/content/sps_evolution_demo")
demo_root.mkdir(exist_ok=True)

registry = demo_root / "registry.json"
events = demo_root / "events.json"

registry.write_text(
    json.dumps({
        "version": "1.0.0",
        "capabilities": [],
        "usage_history": []
    }),
    encoding="utf-8"
)

store = EvolutionEvidenceStore(events, registry)

request = "Create a reusable capability for repeated structured-data parsing failures."
code = "def parse(data): return data"

# Three pieces of disagreement evidence
for n in range(1, 4):
    event = store.record_disagreement(
        session_id="evolution-demo",
        turn_id=n,
        request=request,
        language="python",
        language_confidence=0.95,
        previous_capability_id="CAP-002",
        code=code,
    )

    analysis = store.analyze(event)

    print(
        f"Evidence {n}: "
        f"decision={analysis['decision']}, "
        f"reason={analysis['reason_code']}"
    )

# Actually create the evolved capability
creation = store.record_creation(analysis)

print("\n=== CAPABILITY CREATED ===")
print("Capability ID:", creation["created_capability_id"])
print("Name:", creation["capability_name"])
print("Validation:", creation["validation_status"])

lineage = store.get_capability_lineage(
    creation["created_capability_id"]
)

print("\n=== LINEAGE ===")
print(json.dumps(lineage, indent=2))